# SecureFace-RX — 웹 서버 (코랩)

카메라 없이 `demo.mp4` 를 입력으로 실시간 모니터링 / 보호 자산 / 복원 UI 를 띄웁니다.

**흐름**: demo.mp4 재생 → 얼굴 익명화 → 5초마다 PSF 저장 → `/assets` 에서 복원

**주의**
- 코랩엔 카메라가 없으므로 자동으로 demo.mp4 폴백이 동작합니다.
- 실시간 영상은 프록시를 거쳐 다소 느릴 수 있습니다 (저장/복원 확인엔 충분).
- 셀을 위에서부터 차례로 실행하세요.

## 1. 코드 받기

In [ ]:
!git clone https://github.com/ins420/scrfd-proface.git
%cd scrfd-proface
!git pull
!ls

## 2. 패키지 설치
추론(insightface) + 웹서버(fastapi/uvicorn/jinja2) 모두 설치. CPU onnxruntime 사용.

In [ ]:
!pip install -q insightface onnxruntime opencv-python-headless pycryptodome
!pip install -q fastapi uvicorn jinja2 python-multipart
print('설치 완료')

## 3-1. demo.mp4 업로드

In [ ]:
from google.colab import files
import os
print('▶ demo.mp4 를 선택하세요')
up = files.upload()
for fn in up:
    print(f'  업로드됨: {fn}  ({os.path.getsize(fn)//1024} KB)')

## 3-2. INN 체크포인트(.pth) 업로드

In [ ]:
from google.colab import files
import shutil, os
os.makedirs('checkpoints', exist_ok=True)
print('▶ .pth 파일을 선택하세요')
up = files.upload()
for fn in up:
    if fn.endswith('.pth'):
        shutil.move(fn, f'checkpoints/{fn}')
        print(f'  {fn}  →  checkpoints/')
import config as c
print('INN_CHECKPOINT =', c.INN_CHECKPOINT, '| 존재:', os.path.exists(c.INN_CHECKPOINT))

## 4. 서버 실행 + 접속 URL 생성
백그라운드로 서버를 띄우고, 모델 로드(약 40초)를 기다린 뒤 접속 URL을 출력합니다.

In [ ]:
import subprocess, time

# 기존 서버가 떠 있으면 정리
subprocess.run(['pkill', '-f', 'main.py'])
time.sleep(1)

logf = open('server.log', 'w')
proc = subprocess.Popen(['python', 'main.py'], stdout=logf, stderr=subprocess.STDOUT)
print('서버 시작 중... 모델 로드 + demo.mp4 폴백 대기 (약 45초)')
time.sleep(45)

print('\n===== 서버 로그 (최근 30줄) =====')
!tail -n 30 server.log

In [ ]:
from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(8000)')
print('실시간 모니터링 :', url)
print('보호 자산/복원  :', url + 'assets')
print('\n복원 비밀번호: forensic2026')

## 5. 사용
1. 위 **실시간 모니터링** URL 클릭 → 영상이 재생되며 얼굴 익명화 확인
2. 10~20초 기다리기 (5초마다 PSF 저장됨)
3. **보호 자산/복원** URL → 청크 클릭 → 프레임 클릭 → 비밀번호 `forensic2026` → 복원 확인

### 서버 로그 다시 보기

In [ ]:
!tail -n 40 server.log

### 서버 중지

In [ ]:
import subprocess
subprocess.run(['pkill', '-f', 'main.py'])
print('서버 중지됨')